# DataCo Supply Chain Analysis

This notebook analyzes delivery performance and short-term demand using the DataCo Supply Chain dataset.

## Scope
- Exclude canceled shipments from operational analysis
- Compare late-delivery performance across shipping modes, regions, categories, and time
- Validate the dataset structure before forecasting
- Forecast daily `Order Item Quantity`
- Compare a 7-day moving-average baseline with Prophet
- Produce a 7-day operational demand forecast

> Note: The dataset structure changes on 2017-10-03. Forecast modeling therefore uses data only through 2017-10-02.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Adjust the path if your file is stored elsewhere.
df_law = pd.read_parquet('DataCo_preprocessed.parquet')

print(df_law.shape)
df_law.head()


## 1. Delivery Performance Analysis

Canceled shipments are excluded because the objective is to evaluate completed/active logistics operations.


In [ ]:
df_analysis = df_law[
    df_law['Delivery Status'] != 'Shipping canceled'
].copy()

overall_late_rate = df_analysis['Late_delivery_risk'].mean()

print(f"Overall late delivery rate: {overall_late_rate:.2%}")


In [ ]:
shipping_mode_summary = (
    df_analysis
    .groupby('Shipping Mode')
    .agg(
        records=('Order Id', 'size'),
        late_rate=('Late_delivery_risk', 'mean'),
        scheduled_days=('Days for shipment (scheduled)', 'mean'),
        actual_days=('Days for shipping (real)', 'mean')
    )
    .sort_values('late_rate', ascending=False)
)

shipping_mode_summary['shipping_gap'] = (
    shipping_mode_summary['actual_days']
    - shipping_mode_summary['scheduled_days']
)

shipping_mode_summary


In [ ]:
region_summary = (
    df_analysis
    .groupby('Order Region')
    .agg(
        records=('Order Id', 'size'),
        late_rate=('Late_delivery_risk', 'mean')
    )
    .sort_values('late_rate', ascending=False)
)

category_summary = (
    df_analysis
    .groupby('Category Name')
    .agg(
        records=('Order Id', 'size'),
        late_rate=('Late_delivery_risk', 'mean')
    )
    .sort_values('late_rate', ascending=False)
)

region_summary.head(10), category_summary.head(10)


### Delivery analysis interpretation

Shipping Mode shows the largest variation in late-delivery performance.  
Region and category differences are comparatively small.  
Larger scheduled-versus-actual shipping gaps are associated with higher late-delivery rates, but this analysis does not establish causality.


## 2. Date Parsing and Data Quality Validation

The order-date column contains two different date formats:

- `1/13/2018 12:27` → month/day/year
- `2018. 12. 1 오후 11:51:00` → year/day/month

They must be parsed separately.


In [ ]:
date_text = df_law['order date (DateOrders)']

korean_mask = date_text.str.contains('오전|오후', na=False)

df_law['order datetime'] = pd.NaT

# Standard format
df_law.loc[~korean_mask, 'order datetime'] = pd.to_datetime(
    date_text[~korean_mask],
    format='%m/%d/%Y %H:%M',
    errors='coerce'
)

# Korean AM/PM format
korean_dates = (
    date_text[korean_mask]
    .str.replace('오전', 'AM', regex=False)
    .str.replace('오후', 'PM', regex=False)
)

df_law.loc[korean_mask, 'order datetime'] = pd.to_datetime(
    korean_dates,
    format='%Y. %d. %m %p %I:%M:%S',
    errors='coerce'
)

df_law['order day'] = df_law['order datetime'].dt.date

print("Date parsing failures:", df_law['order datetime'].isna().sum())


In [ ]:
# Re-create analysis data after date preprocessing
df_analysis = df_law[
    df_law['Delivery Status'] != 'Shipping canceled'
].copy()

daily_compare = (
    df_analysis
    .groupby('order day')
    .agg(
        row_count=('Order Id', 'size'),
        order_count=('Order Id', 'nunique')
    )
)

daily_compare['rows_per_order'] = (
    daily_compare['row_count']
    / daily_compare['order_count']
)

first_one_day = daily_compare[
    daily_compare['rows_per_order'] == 1
].index.min()

print("First date with exactly one row per observed order:", first_one_day)

daily_compare.loc[
    pd.Timestamp('2017-09-28').date():
    pd.Timestamp('2017-10-08').date()
]


### Data quality finding

From 2017-10-03 onward, each observed order is represented by exactly one row.  
Before that date, an order can span multiple rows.

Because row-level quantity is affected by this representation change, the post-2017-10-02 period is not directly comparable for the demand-forecasting target.


## 3. Daily Demand Target

The forecasting target is the daily sum of `Order Item Quantity`, excluding canceled shipments.


In [ ]:
daily_quantity = (
    df_analysis
    .groupby('order day')['Order Item Quantity']
    .sum()
)

quantity_before_change = daily_quantity.loc[
    pd.Timestamp('2015-01-01').date():
    pd.Timestamp('2017-10-02').date()
]

print(quantity_before_change.describe())

quantity_before_change.plot(
    figsize=(14, 5),
    title='Daily Order Item Quantity Before Data Structure Change'
)
plt.xlabel('Date')
plt.ylabel('Total Item Quantity')
plt.show()


## 4. Train/Test Split

- Train: through 2017-06-30
- Test: 2017-07-01 through 2017-10-02

The test period contains 94 days.


In [ ]:
train = quantity_before_change.loc[
    :pd.Timestamp('2017-06-30').date()
]

test = quantity_before_change.loc[
    pd.Timestamp('2017-07-01').date():
]

print("Train:", train.index.min(), "to", train.index.max(), "|", len(train), "days")
print("Test :", test.index.min(), "to", test.index.max(), "|", len(test), "days")


## 5. 7-Day Moving-Average Baseline

Each day's prediction uses the previous 7 actual days.


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

full_quantity = pd.concat([train, test])

baseline_pred = (
    full_quantity
    .shift(1)
    .rolling(7)
    .mean()
)

baseline_result = pd.DataFrame({
    'actual': test,
    'predicted': baseline_pred.loc[test.index]
})

baseline_mae = mean_absolute_error(
    baseline_result['actual'],
    baseline_result['predicted']
)

baseline_rmse = mean_squared_error(
    baseline_result['actual'],
    baseline_result['predicted']
) ** 0.5

print(f"7-day baseline MAE : {baseline_mae:.2f}")
print(f"7-day baseline RMSE: {baseline_rmse:.2f}")


## 6. Prophet Walk-Forward Validation

For a fair comparison with the daily-updated baseline:

1. Train Prophet on all data available up to the previous day
2. Predict the next day
3. Add the actual value to history
4. Retrain and repeat


In [ ]:
from prophet import Prophet

prophet_walk_preds = []
history = train.copy()

for date, actual in test.items():
    history_prophet = history.reset_index()
    history_prophet.columns = ['ds', 'y']
    history_prophet['ds'] = pd.to_datetime(history_prophet['ds'])

    model = Prophet()
    model.fit(history_prophet)

    future = pd.DataFrame({
        'ds': [pd.to_datetime(date)]
    })

    forecast = model.predict(future)
    predicted = forecast['yhat'].iloc[0]

    prophet_walk_preds.append(predicted)

    # Add the observed value before predicting the next day
    history.loc[date] = actual

prophet_result = pd.DataFrame({
    'actual': test.values,
    'predicted': prophet_walk_preds
}, index=test.index)

prophet_mae = mean_absolute_error(
    prophet_result['actual'],
    prophet_result['predicted']
)

prophet_rmse = mean_squared_error(
    prophet_result['actual'],
    prophet_result['predicted']
) ** 0.5

print(f"Prophet MAE : {prophet_mae:.2f}")
print(f"Prophet RMSE: {prophet_rmse:.2f}")


In [ ]:
model_comparison = pd.DataFrame({
    'Model': ['7-Day Baseline', 'Prophet'],
    'MAE': [baseline_mae, prophet_mae],
    'RMSE': [baseline_rmse, prophet_rmse]
})

model_comparison


### Model comparison

Prophet slightly outperforms the 7-day moving-average baseline, but the improvement is modest.  
This suggests that the additional time-series structure learned by Prophet provides some value, while a simple recent-history baseline remains competitive.


## 7. Time-Series Diagnostics


In [ ]:
component_forecast = model.predict(
    history_prophet[['ds']]
)

model.plot_components(component_forecast)
plt.show()


In [ ]:
year_month_analysis = quantity_before_change.copy()
year_month_analysis.index = pd.to_datetime(year_month_analysis.index)

year_month_df = year_month_analysis.to_frame(name='quantity')
year_month_df['year'] = year_month_df.index.year
year_month_df['month'] = year_month_df.index.month

year_month_avg = (
    year_month_df
    .groupby(['year', 'month'])['quantity']
    .mean()
    .unstack()
)

plt.figure(figsize=(12, 5))

for year in year_month_avg.index:
    plt.plot(
        year_month_avg.columns,
        year_month_avg.loc[year],
        marker='o',
        label=str(year)
    )

plt.xlabel('Month')
plt.ylabel('Average Daily Quantity')
plt.title('Monthly Average Quantity by Year')
plt.xticks(range(1, 13))
plt.legend()
plt.grid(alpha=0.3)
plt.show()


The weekday effect is relatively weak. The more notable pattern is a declining demand level during 2017.  
Monthly averages should be interpreted cautiously because the pattern is not fully stable across years.


## 8. Final 7-Day Forecast

The final Prophet model is trained on all comparable data through 2017-10-02.


In [ ]:
final_prophet_df = quantity_before_change.reset_index()
final_prophet_df.columns = ['ds', 'y']
final_prophet_df['ds'] = pd.to_datetime(final_prophet_df['ds'])

final_model = Prophet()
final_model.fit(final_prophet_df)

future_7days = final_model.make_future_dataframe(
    periods=7,
    freq='D'
)

final_forecast = final_model.predict(future_7days)

future_result = final_forecast[
    final_forecast['ds'] > final_prophet_df['ds'].max()
][['ds', 'yhat', 'yhat_lower', 'yhat_upper']]

future_result


### Forecast interpretation

The 7-day point forecast is approximately 326–335 units per day, centered around roughly 330 units per day.

Because the dataset structure changes immediately after 2017-10-02, this forecast should be treated as an operational planning scenario rather than as a directly validated post-period forecast.


## 9. Export Results for Power BI


In [ ]:
prophet_result_export = prophet_result.reset_index()
prophet_result_export.columns = ['date', 'actual', 'predicted']
prophet_result_export.to_csv('prophet_result.csv', index=False)

future_result.to_excel(
    'future_forecast_7days.xlsx',
    index=False
)

model_comparison.to_csv(
    'forecast_model_comparison.csv',
    index=False
)

print("Export complete.")


## Key Findings

- Overall late delivery rate is approximately 57.3% after excluding canceled shipments.
- Shipping Mode shows the largest variation in late-delivery performance.
- Region, category, and monthly differences are relatively small by comparison.
- A structural change in the dataset begins on 2017-10-03, so forecasting uses only comparable data through 2017-10-02.
- Prophet achieves slightly lower MAE and RMSE than the 7-day moving-average baseline.
- Short-term demand is forecast to remain around 330 units per day.
